# Two Model-Based Candidate Agents

This notebook focuses only on the two agents selected for fine-tuning/model-based work:

1. **Fundamental Agent** — proposed model: Mistral Small 4; uses `data/silver/silver_edgar_<TICKER>.csv`.
2. **Market + Volatility + News Sentiment Agent** — proposed model-based candidate; uses `data/silver/silver_prices_<TICKER>.csv` and `data/silver/silver_news_sentiment.csv`.

The current code uses deterministic scoring placeholders so the notebook can run now. Later, the scoring internals can be replaced by fine-tuned model inference while preserving the same JSON output format.

Metric note: this notebook reports **agreement with the existing Gold risk label**, not true real-world accuracy. True accuracy needs external labels or future realized outcomes.

In [ ]:
import csv
import json
from collections import Counter
from pathlib import Path

from agents.fundamental_agent import FundamentalAnalysisAgent
from agents.market_sentiment_agent import MarketSentimentAgent

GOLD_PATH = Path('data/gold/gold_risk_scores_ALL.csv')
OUTPUT_PATH = Path('data/gold/two_agent_evaluation_results.csv')

## 1. Run Both Agents For One Ticker

In [ ]:
ticker = 'KO'

fundamental_agent = FundamentalAnalysisAgent()
market_sentiment_agent = MarketSentimentAgent()

fundamental_result = fundamental_agent.analyze(ticker)
market_sentiment_result = market_sentiment_agent.analyze(ticker)

print('FUNDAMENTAL AGENT')
print(json.dumps(fundamental_result.__dict__, indent=2))

print('\nMARKET + VOLATILITY + NEWS SENTIMENT AGENT')
print(json.dumps(market_sentiment_result.__dict__, indent=2))

## 2. Load Existing Gold Labels

In [ ]:
def load_gold_labels(path):
    with open(path, newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    return [
        {
            'ticker': row['ticker'].upper(),
            'gold_label': row['risk_label'],
            'gold_score': float(row['composite_risk']),
        }
        for row in rows
        if row.get('ticker') and row.get('risk_label')
    ]

gold_rows = load_gold_labels(GOLD_PATH)
print(f'Loaded {len(gold_rows)} Gold rows')
print(Counter(row['gold_label'] for row in gold_rows))

## 3. Evaluate The Two Agents Across All Gold Tickers

In [ ]:
def safe_agent_result(fn, ticker):
    try:
        result = fn(ticker)
        return result.risk_label, result.risk_score, result.confidence, ''
    except Exception as exc:
        return 'ERROR', None, 'ERROR', str(exc)

evaluation_rows = []
for row in gold_rows:
    ticker = row['ticker']
    gold_label = row['gold_label']

    fundamental_label, fundamental_score, fundamental_confidence, fundamental_error = safe_agent_result(
        fundamental_agent.analyze, ticker
    )
    market_sentiment_label, market_sentiment_score, market_sentiment_confidence, market_sentiment_error = safe_agent_result(
        market_sentiment_agent.analyze, ticker
    )

    evaluation_rows.append({
        'ticker': ticker,
        'gold_label': gold_label,
        'gold_score': row['gold_score'],
        'fundamental_label': fundamental_label,
        'fundamental_score': fundamental_score,
        'fundamental_confidence': fundamental_confidence,
        'fundamental_match': fundamental_label == gold_label,
        'fundamental_error': fundamental_error,
        'market_sentiment_label': market_sentiment_label,
        'market_sentiment_score': market_sentiment_score,
        'market_sentiment_confidence': market_sentiment_confidence,
        'market_sentiment_match': market_sentiment_label == gold_label,
        'market_sentiment_error': market_sentiment_error,
    })

print(f'Evaluated {len(evaluation_rows)} tickers')

## 4. Agreement Summary

In [ ]:
def agreement(rows, match_col, label_col):
    usable = [row for row in rows if row[label_col] != 'ERROR']
    matches = [row for row in usable if row[match_col]]
    return {
        'usable': len(usable),
        'matches': len(matches),
        'agreement': round(len(matches) / len(usable), 3) if usable else 0.0,
        'label_counts': dict(Counter(row[label_col] for row in usable)),
    }

summary = {
    'fundamental_agent': agreement(evaluation_rows, 'fundamental_match', 'fundamental_label'),
    'market_sentiment_agent': agreement(evaluation_rows, 'market_sentiment_match', 'market_sentiment_label'),
}

print(json.dumps(summary, indent=2))

## 5. Save Evaluation CSV

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
fieldnames = list(evaluation_rows[0].keys()) if evaluation_rows else []
with open(OUTPUT_PATH, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(evaluation_rows)

print(f'Saved {len(evaluation_rows)} rows to {OUTPUT_PATH}')

## 6. Inspect Mismatches

In [ ]:
agent = 'market_sentiment'  # change to: fundamental or market_sentiment
label_col = f'{agent}_label'
match_col = f'{agent}_match'
score_col = f'{agent}_score'

mismatches = [row for row in evaluation_rows if row[label_col] != 'ERROR' and not row[match_col]]
for row in mismatches[:20]:
    print(row['ticker'], 'gold=', row['gold_label'], 'agent=', row[label_col], 'score=', row.get(score_col))

print(f'Total mismatches for {agent}: {len(mismatches)}')